## Libraries

In [0]:
from pyspark.sql.functions import col, avg, when
from pyspark.sql import DataFrame
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from pyspark.ml import Pipeline
from pyspark.sql.functions import concat_ws
import pandas as pd

## Load data

In [0]:
# Read encoded (or indexed) data
indexed_data = spark.table('workspace.telco.indexed_data')
display(indexed_data.limit(5))

## Features

I'll create two approach to create new features.

1- Statistical features from numerical features grouping by categorical columns. This new features are:
  - Mean of monthly charge.
  - Mean of total charge.
  - Mean of tenure.
  - Mean Monthly charge-tenure ratio.
  - Mean Total charge-tenure ratio.

2- A combination of the categorical features to expose or create new relationships between to categorical columns.

### Numerical features

In [0]:
# Categorical columns
categorical_cols = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

In [0]:
indexed_data_df = indexed_data.toPandas() # Convert to pandas dataframe
# Creating a new tenure columns to avoid division by zero
indexed_data_df["tenure_safe"] = indexed_data_df["tenure"].apply(lambda x: x if x > 0 else None) 

def feat_related_charges(df: pd.DataFrame, column:str):
    """
        New features related to charges
        Args:
            df: pandas dataframe
            column: categorical column name
        Returns:
            New dataframe with new features
    """
    # Monthly charge-tenure ratio
    df["monthly_charge_tenure"] = df["MonthlyCharges"] / df["tenure_safe"]
    # Total charge-tenure ratio
    df["total_charge_tenure"] = df["TotalCharges"] / df["tenure_safe"]
    
    # Mean tenure by category
    df[column + "_mean_tenure"] = df.groupby(column)["tenure"].transform("mean")
    # Mean monthly charge by category
    df[column + "_mean_monthly_charge"] = df.groupby(column)["MonthlyCharges"].transform("mean")
    # Mean total charge by category
    df[column + "_mean_total_charge"] = df.groupby(column)["TotalCharges"].transform("mean")

    # Mean monthly charge-tenure ratio by category    
    df[column + "_mean_monthly_charge_tenure"] = df.groupby(column)["monthly_charge_tenure"].transform("mean")
    # Mean total charge-tenure ratio by category
    df[column + "_mean_total_charge_tenure"] = df.groupby(column)["total_charge_tenure"].transform("mean")

    return df

for column in categorical_cols:
    indexed_data_df = feat_related_charges(indexed_data_df, column)
indexed_data_df.head()

## Combining categorical features

In [0]:
# Identify categorical columns
for col in categorical_cols:
    indexed_data_df[col] = indexed_data_df[col].astype("object")
indexed_data_df.info()

In [0]:
for col1 in categorical_cols: # Iterate over categorical columns
    for col2 in categorical_cols: # Iterate over categorical columns
        if col1 != col2: # If the columns are not the same
            # Create a new column
            indexed_data_df[f"{col1}_{col2}"] = (
                indexed_data_df[col1].astype(str) + "_" + indexed_data_df[col2].astype(str)
            )
indexed_data_df.head()

## Encoding new categorical fields

In [0]:
# Identify all the new categorical features
new_categorical_names = []
for col1 in categorical_cols:
    for col2 in categorical_cols:
        if col1 != col2:
            new_categorical_names.append(f"{col1}_{col2}")
print(new_categorical_names)

In [0]:
# Encoder
le = LabelEncoder()
# Fit and transform
for col in new_categorical_names:
    indexed_data_df[col] = le.fit_transform(indexed_data_df[col])
indexed_data_df.head()

In [0]:
# Drop tenure_safe
indexed_data_df.drop(["tenure_safe"], inplace=True, axis=1)

## Scaling numerical features

In [0]:
# Selecting numerical features
numerical_cols = indexed_data_df.select_dtypes(include=[int, float]).columns
# Adding new numerical features
final_num_cols = list(set(numerical_cols) - set(categorical_cols) - set(new_categorical_names) - set(["Churn"]))
final_num_cols

In [0]:
# Scaler
scaler = MinMaxScaler()
# Scale the data
scaled_data = pd.DataFrame(scaler.fit_transform(indexed_data_df[final_num_cols]))
scaled_data.columns = final_num_cols # Set columns name
scaled_data.head()

In [0]:
# Drop non-scaled columns
indexed_tmp = indexed_data_df.drop(final_num_cols, axis=1)
# Concatenates scaled data
scaled_data = pd.concat([indexed_tmp, scaled_data], axis=1)

In [0]:
# Saving scaling and encoding data
scaled_data = spark.createDataFrame(scaled_data)
scaled_data.write.mode("overwrite").saveAsTable("workspace.telco.ml_silver_data")